In [1]:
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation, PillowWriter

from kerr_schild_geodesics import (
    normalization,
    energy_lz,
)

from kerr_schild_geodesics.integrators import (
    integrate_rk4,
)


# ============================================================
# Schwarzschild parameters
# ============================================================

M = 1.0
a = 0.0


# ============================================================
# Bound timelike orbit parameters
# ============================================================

# Semi-latus rectum
p = 8.0

# Eccentricity
e = 0.20

# Tilt the orbital plane for a useful 3D view.
inclination = np.deg2rad(35.0)


# ------------------------------------------------------------
# Exact Schwarzschild bound-orbit constants
# ------------------------------------------------------------

E_squared = (
    (p - 2.0 - 2.0*e)
    * (p - 2.0 + 2.0*e)
    / (
        p
        * (p - 3.0 - e**2)
    )
)

L_squared = (
    p**2
    * M**2
    / (
        p - 3.0 - e**2
    )
)

E = np.sqrt(E_squared)
L = np.sqrt(L_squared)


# Pericenter and apocenter
r_peri = (
    p * M
    / (1.0 + e)
)

r_apo = (
    p * M
    / (1.0 - e)
)


print("E =", E)
print("L =", L)

print(
    "pericenter =",
    r_peri,
    "M"
)

print(
    "apocenter =",
    r_apo,
    "M"
)

E = 0.9503819266229828
L = 3.5921060405354983
pericenter = 6.666666666666667 M
apocenter = 10.0 M


In [2]:
# ============================================================
# Initial condition at apocenter
# ============================================================

r0 = r_apo

x0 = r0
y0 = 0.0
z0 = 0.0


# Schwarzschild u^t at the turning point
ut0 = (
    E
    / (
        1.0
        - 2.0*M/r0
    )
)


# Angular motion
u_phi = (
    L
    / r0**2
)


# Tangential Cartesian four-velocity magnitude
u_tangent = (
    r0
    * u_phi
)


# Tilt the orbital plane around the x axis
ux0 = 0.0

uy0 = (
    u_tangent
    * np.cos(inclination)
)

uz0 = (
    u_tangent
    * np.sin(inclination)
)


state0 = np.array([
    0.0,     # t
    x0,
    y0,
    z0,
    ut0,
    ux0,
    uy0,
    uz0,
])


print(
    "initial normalization =",
    normalization(
        state0,
        M,
        a
    )
)

print(
    "initial E, Lz =",
    energy_lz(
        state0,
        M,
        a
    )
)

initial normalization = -0.9999999999999996
initial E, Lz = (0.9503819266229828, 2.9424810064074896)


In [3]:
# ============================================================
# Integrate Schwarzschild bound geodesic
# ============================================================

h = 0.02

lambda_end = 1800.0

n_steps = int(
    lambda_end / h
)


trajectory_schw = integrate_rk4(
    state0,
    h,
    n_steps,
    M,
    a
)


print(
    "RK4 steps =",
    n_steps
)

RK4 steps = 90000


In [4]:
# ============================================================
# Diagnostics
# ============================================================

sample = trajectory_schw[::500]


r_values = np.sqrt(
    sample[:, 1]**2
    + sample[:, 2]**2
    + sample[:, 3]**2
)


norm_values = np.array([
    normalization(
        state,
        M,
        a
    )
    for state in sample
])


EL_values = np.array([
    energy_lz(
        state,
        M,
        a
    )
    for state in sample
])


print(
    "radial range =",
    r_values.min(),
    "to",
    r_values.max()
)

print(
    "max |norm + 1| =",
    np.max(
        np.abs(
            norm_values + 1.0
        )
    )
)

print(
    "max |E - E0| =",
    np.max(
        np.abs(
            EL_values[:, 0]
            - EL_values[0, 0]
        )
    )
)

print(
    "max |Lz - Lz0| =",
    np.max(
        np.abs(
            EL_values[:, 1]
            - EL_values[0, 1]
        )
    )
)

radial range = 6.669912892014887 to 10.0
max |norm + 1| = 5.784261958297066e-13
max |E - E0| = 2.1826984664130578e-13
max |Lz - Lz0| = 1.234568003383174e-13


In [6]:
# ============================================================
# Smooth Schwarzschild animation with event horizon
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FIGURE_DIR = PROJECT_ROOT / "figures"
FIGURE_DIR.mkdir(exist_ok=True)

print("Saving figures to:", FIGURE_DIR)

anim_data = trajectory_schw[::100]

xv = anim_data[:, 1]
yv = anim_data[:, 2]
zv = anim_data[:, 3]


# ============================================================
# Schwarzschild event horizon
# ============================================================

# For Schwarzschild:
#
# r_H = 2M
#
horizon_radius = 2.0 * M


theta = np.linspace(
    0.0,
    np.pi,
    24
)

phi = np.linspace(
    0.0,
    2.0*np.pi,
    48
)

theta_grid, phi_grid = np.meshgrid(
    theta,
    phi
)


# Schwarzschild horizon is a sphere
xh = (
    horizon_radius
    * np.sin(theta_grid)
    * np.cos(phi_grid)
)

yh = (
    horizon_radius
    * np.sin(theta_grid)
    * np.sin(phi_grid)
)

zh = (
    horizon_radius
    * np.cos(theta_grid)
)


# ============================================================
# Figure
# ============================================================

fig = plt.figure(
    figsize=(7, 7)
)

ax = fig.add_subplot(
    111,
    projection="3d"
)


# Scientific visualization:
# remove perspective distortion
ax.set_proj_type("ortho")


# ------------------------------------------------------------
# Full trajectory in background
# ------------------------------------------------------------

ax.plot(
    xv,
    yv,
    zv,
    alpha=0.18,
    linewidth=0.7
)


# ------------------------------------------------------------
# Schwarzschild event horizon
# ------------------------------------------------------------

ax.plot_surface(
    xh,
    yh,
    zh,
    alpha=0.70,
    linewidth=0.0,
    antialiased=True
)


# ------------------------------------------------------------
# Moving orbit tail
# ------------------------------------------------------------

moving_line, = ax.plot(
    [],
    [],
    [],
    linewidth=1.35
)


# ------------------------------------------------------------
# Moving particle
# ------------------------------------------------------------

moving_point, = ax.plot(
    [],
    [],
    [],
    marker="o",
    linestyle="None",
    markersize=5
)


# ============================================================
# Labels
# ============================================================

ax.set_xlabel(r"$x/M$")
ax.set_ylabel(r"$y/M$")
ax.set_zlabel(r"$z/M$")

ax.set_title(
    "Bound Schwarzschild geodesic"
)


# ============================================================
# Axis limits
# ============================================================

pad = 0.5

xmin = xv.min() - pad
xmax = xv.max() + pad

ymin = yv.min() - pad
ymax = yv.max() + pad

zmin = zv.min() - pad
zmax = zv.max() + pad


ax.set_xlim(
    xmin,
    xmax
)

ax.set_ylim(
    ymin,
    ymax
)

ax.set_zlim(
    zmin,
    zmax
)


# ------------------------------------------------------------
# Equal physical scale
# ------------------------------------------------------------

x_range = xmax - xmin
y_range = ymax - ymin
z_range = zmax - zmin


ax.set_box_aspect(
    (
        x_range,
        y_range,
        z_range
    )
)


# ============================================================
# Animation settings
# ============================================================

fps = 15

n_frames = min(
    450,
    len(xv)
)

frame_indices = np.linspace(
    1,
    len(xv) - 1,
    n_frames,
    dtype=int
)

tail = 90


def update(frame_number):

    i = frame_indices[
        frame_number
    ]

    j = max(
        0,
        i - tail
    )

    # Moving trajectory
    moving_line.set_data(
        xv[j:i+1],
        yv[j:i+1]
    )

    moving_line.set_3d_properties(
        zv[j:i+1]
    )

    # Moving particle
    moving_point.set_data(
        [xv[i]],
        [yv[i]]
    )

    moving_point.set_3d_properties(
        [zv[i]]
    )

    return (
        moving_line,
        moving_point
    )


# ============================================================
# Create animation
# ============================================================

animation = FuncAnimation(
    fig,
    update,
    frames=n_frames,
    interval=1000/fps,
    blit=False
)


# ============================================================
# Save
# ============================================================

gif_path = (
    FIGURE_DIR
    / "bound_schwarzschild_orbit.gif"
)


animation.save(
    gif_path,
    writer=PillowWriter(
        fps=fps
    )
)


plt.close(fig)


print(
    "saved:",
    gif_path
)

print(
    "Schwarzschild horizon radius =",
    horizon_radius,
    "M"
)

print(
    "animation duration =",
    n_frames/fps,
    "seconds"
)

Saving figures to: /home/allanur/Desktop/kerr-schild-geodesics/figures
saved: /home/allanur/Desktop/kerr-schild-geodesics/figures/bound_schwarzschild_orbit.gif
Schwarzschild horizon radius = 2.0 M
animation duration = 30.0 seconds
